# Lab 5: Central Gateway를 사용하는 Supervisor Agent

## 개요
AgentCore Runtime에 배포되어 Central Gateway를 통해 세 가지 전문 Agent를 오케스트레이션하는 Supervisor Agent를 구축합니다.
- **진단 Agent** (Lab 02의 Lambda)
- **문제 해결 Agent** (Lab 03a의 Runtime)
- **예방 Agent** (Lab 04의 Runtime)

## 아키텍처
```
┌─────────────────────────────────────────────────────────────────────┐
│                              USER                                   │
│                    (Cognito JWT Token).                             │
└────────────────────────────┬────────────────────────────────────────┘
                             │
                             │ HTTP POST /invocations
                             │ Authorization: Bearer <JWT>
                             ↓
┌─────────────────────────────────────────────────────────────────────┐
│                    SUPERVISOR AGENT (Local)                         │
│                                                                     │
│  • Orchestrates incident response workflow                          │
│  • Coordinates 3 specialized agents via MCP                         │
└────────────────────────────┬────────────────────────────────────────┘
                             │
                             │ MCP with JWT (passes Authorization header)
                             ↓
┌─────────────────────────────────────────────────────────────────────┐
│                    CENTRAL GATEWAY (MCP)                            │
│  • Single gateway for all agent communication                       │
│  • JWT Authorization (Cognito)                                      │
│  • Lambda Interceptor at REQUEST phase                              │
└────────────────────────────┬────────────────────────────────────────┘
                             │
                             │ Interceptor validates JWT
                             ↓
┌─────────────────────────────────────────────────────────────────────┐
│                    LAMBDA INTERCEPTOR (Lab 03b)                     │
│  • Extracts cognito:groups from JWT                                 │
│  • Approvers: Full access to all tools                              │
│  • SRE: Limited access (diagnostics + prevention only)              │
└────────────────────────────┬────────────────────────────────────────┘
                             │
                    ┌────────┴────────┐
                    ↓                 ↓
                 ALLOW              DENY
                    ↓                 ↓
         Routes to targets    Returns error
                    │
    ┌───────────────┼───────────────┬───────────────┐
    │               │               │               │
    ↓               ↓               ↓               ↓
┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐
│ TARGET 1│   │ TARGET 2│   │ TARGET 3│   │ (Future)│
│         │   │         │   │         │   │         │
│ Lambda  │   │ Runtime │   │ Runtime │   │ Agents  │
│ (Lab 02)│   │(Lab 03a)│   │ (Lab 04)│   │         │
│         │   │         │   │         │   │         │
│Diagnose │   │Remediate│   │ Prevent │   │   ...   │
└─────────┘   └─────────┘   └─────────┘   └─────────┘
     │             │             │
     │ IAM Role    │ OAuth2 M2M  │ OAuth2 M2M
     │             │             │
     ↓             ↓             ↓
┌─────────────────────────────────────────────────────────────────────┐
│                    AWS INFRASTRUCTURE                               │
│  • CloudWatch Logs & Metrics                                        │
│  • DynamoDB Tables                                                  │
│  • EC2 Instances                                                    │
│  • AgentCore Code Interpreter                                       │
│  • AgentCore Browser                                                │
└─────────────────────────────────────────────────────────────────────┘
```

### 주요 구성 요소

1. **Supervisor Agent(Runtime)**
   - JWT가 포함된 `/invocations` HTTP POST 요청을 수신합니다.
   - Authorization 헤더를 추출하여 MCP 클라이언트에 전달합니다.
   - 모든 전문 Agent를 오케스트레이션합니다.

2. **중앙 Gateway**
   - 모든 Agent 통신에 단일 Gateway를 사용합니다.
   - Cognito를 통해 JWT 인증을 수행합니다.
   - Lambda 인터셉터가 토큰을 검증합니다.

3. **Lambda 인터셉터 (Lab 03b)**
   - Lab 03b에서 만든 인터셉터를 재사용합니다.
   - JWT를 검증하고 cognito:groups를 확인합니다.
   - 역할 기반 권한을 적용합니다.

4. **세 가지 전문 Agent**
   - **진단** (Lambda): 로그와 지표를 분석합니다.
   - **문제 해결** (Runtime): Code Interpreter로 문제 해결 작업을 실행합니다.
   - **예방** (Runtime): Browser로 모범 사례를 조사합니다.

### 인증 흐름

```
User → Cognito (JWT)
         ↓
HTTP POST /invocations (Authorization: Bearer <JWT>)
         ↓
Supervisor extracts Authorization header
         ↓
MCP Client → Central Gateway (with Authorization header)
         ↓
Interceptor validates JWT and cognito:groups
         ↓
┌────────┴────────┐
↓                 ↓
Approvers         SRE
(All tools)       (Diagnostics + Prevention only)
```

## 사전 요구 사항
- ✅ Lab 02 완료 (진단 Lambda)
- ✅ Lab 03a 완료 (문제 해결 Runtime)
- ✅ Lab 03b 완료 (인터셉터 Lambda)
- ✅ Lab 04 완료 (예방 Runtime)

## 0. 종속성 설치

In [ ]:
%pip install -q -r requirements.txt
print("✅ Dependencies installed")

## 1. 모듈 가져오기 및 구성

In [ ]:
import json
import boto3
import time
import urllib.parse

from lab_helpers.config import AWS_REGION, WORKSHOP_NAME
from lab_helpers.parameter_store import get_parameter, put_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.lab_01.infrastructure import get_app_url

print("✅ Imports loaded")
print(f"   Region: {AWS_REGION}")
print(f"   Workshop: {WORKSHOP_NAME}")

## 2. 사전 요구 사항 확인

In [ ]:
# 필요한 리소스가 모두 존재하는지 확인
try:
    lab02_lambda_arn = get_parameter(PARAMETER_PATHS["lab_02"]["lambda_function_arn"])
    lab03_runtime_arn = get_parameter(PARAMETER_PATHS["lab_03"]["runtime_arn"])
    lab04_runtime_arn = get_parameter(PARAMETER_PATHS["lab_04"]["runtime_arn"])
    interceptor_arn = get_parameter(PARAMETER_PATHS["lab_03b"]["interceptor_function_arn"])
    user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])

    print("✅ All prerequisites verified")
    print(f"   Lab 02 Lambda: {lab02_lambda_arn}")
    print(f"   Lab 03a Runtime: {lab03_runtime_arn}")
    print(f"   Lab 04 Runtime: {lab04_runtime_arn}")
    print(f"   Interceptor: {interceptor_arn}")
    print(f"   Cognito Pool: {user_pool_id}")
except Exception as e:
    print(f"❌ Missing prerequisites: {e}")
    raise

## 3. 인터셉터가 적용된 Central Gateway 생성

In [ ]:
# Gateway 서비스 역할 생성
iam = boto3.client("iam", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)
account_id = sts.get_caller_identity()["Account"]

gateway_role_name = f"{WORKSHOP_NAME}_CentralGatewayRole"
gateway_trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
gateway_permissions = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": "*"},
        {"Effect": "Allow", "Action": "bedrock-agentcore:*", "Resource": "*"},
        {"Effect": "Allow", "Action": "logs:*", "Resource": "*"},
        {
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:*",
                "bedrock:*",
                "agent-credential-provider:*",
                "iam:PassRole",
                "secretsmanager:GetSecretValue",
                "lambda:InvokeFunction",
            ],
            "Resource": "*",
        },
    ],
}

try:
    gw_role = iam.create_role(RoleName=gateway_role_name, AssumeRolePolicyDocument=json.dumps(gateway_trust))
    gateway_role_arn = gw_role["Role"]["Arn"]
    time.sleep(10)
except iam.exceptions.EntityAlreadyExistsException:
    gateway_role_arn = iam.get_role(RoleName=gateway_role_name)["Role"]["Arn"]

iam.put_role_policy(
    RoleName=gateway_role_name,
    PolicyName="GatewayPermissions",
    PolicyDocument=json.dumps(gateway_permissions),
)
print(f"✅ Gateway role: {gateway_role_arn}")

In [ ]:
# Gateway 역할 ARN을 Parameter Store에 저장
put_parameter(
    PARAMETER_PATHS["lab_05"]["gateway_role_arn"],
    gateway_role_arn,
    region_name=AWS_REGION,
)

**JWT 권한 부여를 사용하는 Central Gateway 생성**

다음 구성으로 AgentCore Gateway를 생성합니다.
- 사용자 지정 JWT 권한 부여자 (Cognito 검색 URL)
- 허용된 클라이언트: 사용자 인증 + M2M
- 요청 권한 부여를 위한 Lambda 인터셉터
- Gateway ID와 URL을 Parameter Store에 저장

In [ ]:
# Cognito 구성 가져오기
user_auth_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"])
m2m_client_id = get_parameter(PARAMETER_PATHS["cognito"]["m2m_client_id"])
discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

# 인터셉터가 적용된 Gateway 생성
agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)
gateway = agentcore.create_gateway(
    name="aiml301-central-gateway",
    roleArn=gateway_role_arn,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [user_auth_client_id, m2m_client_id],
        }
    },
    interceptorConfigurations=[
        {
            "interceptionPoints": ["REQUEST"],
            "interceptor": {"lambda": {"arn": interceptor_arn}},
            "inputConfiguration": {"passRequestHeaders": True},
        }
    ],
)

central_gateway_id = gateway["gatewayId"]
central_gateway_url = gateway["gatewayUrl"]
put_parameter(PARAMETER_PATHS["lab_05"]["gateway_id"], central_gateway_id, region_name=AWS_REGION)

put_parameter(
    PARAMETER_PATHS["lab_05"]["gateway_url"],
    central_gateway_url,
    region_name=AWS_REGION,
)

print(f"✅ Gateway created: {central_gateway_id}")
print(f"   URL: {central_gateway_url}")

In [ ]:
!aws sts get-caller-identity

In [ ]:
# Lambda 권한 추가
lambda_client = boto3.client("lambda", region_name=AWS_REGION)
try:
    lambda_client.add_permission(
        FunctionName=interceptor_arn.split(":")[-1],
        StatementId=f"AllowGateway{central_gateway_id[:8]}",
        Action="lambda:InvokeFunction",
        Principal="bedrock-agentcore.amazonaws.com",
        SourceArn=f"arn:aws:bedrock-agentcore:{AWS_REGION}:{account_id}:gateway/{central_gateway_id}",
    )
    print("✅ Lambda permission added")
except lambda_client.exceptions.ResourceConflictException:
    print("ℹ️  Permission exists")

## 4. Target 1 추가: 진단 Lambda

In [ ]:
diagnostics_schema = [
    {
        "name": "logs_analysis_agent",
        "description": "This agent is a system diagnostics tool that analyzes AWS infrastructure health by examining logs and metrics from EC2, NGINX, and DynamoDB services. It uses AI to correlate findings across different data sources, identify issues like throttling, high resource utilization, and application errors, then provides evidence-based assessments with recommended actions for troubleshooting system problems.",
        "inputSchema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    }
]

target1 = agentcore.create_gateway_target(
    gatewayIdentifier=central_gateway_id,
    name="strands-diagnostics-agent",
    targetConfiguration={
        "mcp": {
            "lambda": {
                "lambdaArn": lab02_lambda_arn,
                "toolSchema": {"inlinePayload": diagnostics_schema},
            }
        }
    },
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
)
print(f"✅ Target 1: {target1['targetId']}")

## 5. Target 2 추가: 문제 해결 Runtime

In [ ]:
# OAuth2 공급자 가져오기
try:
    oauth2_provider_arn = get_parameter(PARAMETER_PATHS["lab_03"]["oauth2_provider_arn"])
except:
    m2m_client_secret = get_parameter(PARAMETER_PATHS["cognito"]["m2m_client_secret"])
    cred = agentcore.create_oauth2_credential_provider(
        name="aiml301-m2m-lab05a",
        credentialProviderVendor="CustomOauth2",
        oauth2ProviderConfigInput={
            "customOauth2ProviderConfig": {
                "clientId": m2m_client_id,
                "clientSecret": m2m_client_secret,
                "oauthDiscovery": {"discoveryUrl": discovery_url},
            }
        },
    )
    oauth2_provider_arn = cred["credentialProviderArn"]

resource_server_id = get_parameter(PARAMETER_PATHS["cognito"]["resource_server_identifier"])
m2m_scopes = [
    f"{resource_server_id}/mcp.invoke",
    f"{resource_server_id}/runtime.access",
]

encoded_arn = urllib.parse.quote(lab03_runtime_arn, safe="")
remediation_endpoint = (
    f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
)

target2 = agentcore.create_gateway_target(
    gatewayIdentifier=central_gateway_id,
    name="aiml301-runtime-target",
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": remediation_endpoint}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": oauth2_provider_arn,
                    "scopes": m2m_scopes,
                }
            },
        }
    ],
)
print(f"✅ Target 2: {target2['targetId']}")

## 6. Target 3 추가: 예방 Runtime

In [ ]:
encoded_arn = urllib.parse.quote(lab04_runtime_arn, safe="")
prevention_endpoint = (
    f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
)

target3 = agentcore.create_gateway_target(
    gatewayIdentifier=central_gateway_id,
    name="aiml301-prevention-runtime-target",
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": prevention_endpoint}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": oauth2_provider_arn,
                    "scopes": m2m_scopes,
                }
            },
        }
    ],
)
print(f"✅ Target 3: {target3['targetId']}")

**Gateway Target 동기화**

Target 동기화를 시작하고 다음 상태가 될 때까지 Gateway 상태를 폴링합니다.
- `AVAILABLE`: 동기화 성공
- `FAILED`: 사유와 함께 동기화 실패
- 30회 시도(60초) 후 시간 초과

In [ ]:
time.sleep(20)

In [ ]:
# Runtime Target만 동기화(Lambda 제외)
for target in [target2, target3]:  # 문제 해결 및 예방(Runtime)
    agentcore.synchronize_gateway_targets(gatewayIdentifier=central_gateway_id, targetIdList=[target["targetId"]])
    print(f"⏳ Synchronizing {target['targetId']}...")

# 동기화 상태 폴링
import time

max_attempts = 30
for i in range(max_attempts):
    time.sleep(2)
    response = agentcore.get_gateway(gatewayIdentifier=central_gateway_id)
    status = response["status"]

    if status == "READY":
        print("✅ Runtime targets synchronized successfully")
        break
    elif status == "FAILED":
        print(f"❌ Synchronization failed: {response.get('failureReasons', [])}")
        break
    else:
        print(f"   Status: {status} ({i + 1}/{max_attempts})")
else:
    print("⚠️ Synchronization timeout - check gateway status manually")

## 7. 토큰 가져오기

In [ ]:
# 승인자로 인증
cognito = boto3.client("cognito-idp", region_name=AWS_REGION)
approver_username = get_parameter(PARAMETER_PATHS["cognito"]["approver_user_email"])
approver_password = get_parameter(PARAMETER_PATHS["cognito"]["approver_user_password"])

auth_response = cognito.initiate_auth(
    ClientId=user_auth_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": approver_username, "PASSWORD": approver_password},
)
access_token = auth_response["AuthenticationResult"]["AccessToken"]
print("✅ Authenticated as approver")

In [ ]:
print(access_token)

## 8. Supervisor 코드 실행

**로컬 Supervisor Agent 실행**

Strands 프레임워크를 사용하여 Supervisor Agent를 로컬에서 실행합니다.
- JWT 인증으로 Central Gateway에 연결합니다.
- Gateway Target에서 사용 가능한 모든 도구를 가져옵니다.
- 진단, 문제 해결 및 예방 Agent를 오케스트레이션합니다.
- 사용자 쿼리를 처리하고 종합적인 응답을 반환합니다.

### Supervisor 도구 목록 호출 - 세 Agent가 도구로 노출된 Central Gateway에 연결

In [ ]:
from lab_helpers.lab_05.local_supervisor_agent import run_supervisor_agent

# 쿼리 구성
user_prompt = "List all tools available"

# Supervisor Agent 실행
response = run_supervisor_agent(gateway_url=central_gateway_url, access_token=access_token, prompt=user_prompt)

print("\n" + "=" * 80)
print("SUPERVISOR AGENT RESPONSE")
print("=" * 80)
print(response)

### 진단 Agent 호출

In [ ]:
# 쿼리 구성
user_prompt = "what recent issues related to dynamodb throttling, do you see in the CRM application"

# Supervisor Agent 실행
response = run_supervisor_agent(gateway_url=central_gateway_url, access_token=access_token, prompt=user_prompt)

print("\n" + "=" * 80)
print("SUPERVISOR AGENT RESPONSE")
print("=" * 80)
print(response)

**Streamlit 예제 쿼리**

Supervisor Agent에서 다음 예제 쿼리를 사용해 보세요.

```python
# 진단 query
"What recent issues related to DynamoDB throttling do you see in the CRM application?"

# 인프라 query
"List all DynamoDB tables. Use infrastructure agent and action_type=only_execute"

# 예방 query
"Research DynamoDB best practices to avoid table throttling"

# 전체 workflow
"Fix IAM Permissions to resolve dynamo DB access issue.You have the required IAM permissions and always use action_type='only_execute'"
```

## 9. Streamlit 대화형 채팅 인터페이스

사용자 친화적인 UI를 통해 Supervisor Agent와 상호 작용할 수 있는 웹 기반 채팅 인터페이스를 시작합니다.

### 9.1 Gateway 구성 파일 생성

In [ ]:
# Streamlit 앱용 gateway_config.json 생성
# 참고: GatewayClient에는 Cognito 인증을 위한 특정 필드가 필요함
gateway_config = {
    "gateway_id": central_gateway_id,
    "gateway_url": central_gateway_url,
    "region": AWS_REGION,
    "client_info": {
        "user_pool_id": user_pool_id,
        "client_id": user_auth_client_id,
        "client_secret": "",  # 사용자 인증 클라이언트에서는 비워 둠(보안 암호 없음)
        "scope": "openid",  # Cognito 필수 범위
        "username": approver_username,
        "password": approver_password,
    },
}

import json

with open("gateway_config.json", "w") as f:
    json.dump(gateway_config, f, indent=2)

print("✅ Gateway configuration saved to gateway_config.json")
print(f"   Gateway ID: {central_gateway_id}")
print(f"   Gateway URL: {central_gateway_url}")
print(f"   User: {approver_username}")

### 9.2 Streamlit 설치

In [ ]:
%pip install -q streamlit
print("✅ Streamlit installed")

### 9.3 Streamlit 앱 시작


In [ ]:
import subprocess
import time
import os
import json
import boto3


def get_streamlit_url():
    """
    SageMaker Studio 또는 로컬 환경에서 접근 가능한 Streamlit URL을 생성합니다.
    """
    try:
        # SageMaker Studio에서 실행 중인지 확인
        with open("/opt/ml/metadata/resource-metadata.json", "r") as file:
            data = json.load(file)
            domain_id = data["DomainId"]
            space_name = data["SpaceName"]

        # SageMaker Studio URL 가져오기
        sagemaker_client = boto3.client("sagemaker")
        response = sagemaker_client.describe_space(DomainId=domain_id, SpaceName=space_name)
        streamlit_url = response["Url"] + "/proxy/8501/"
        print("📍 Running in SageMaker Studio")
        print(f"   Domain ID: {domain_id}")
        print(f"   Space Name: {space_name}")

    except (FileNotFoundError, json.JSONDecodeError, KeyError):
        # 로컬 또는 SageMaker Studio 외부에서 실행 중
        streamlit_url = "http://localhost:8501"
        print("📍 Running in local environment")

    return streamlit_url


# 기존 Streamlit 프로세스 종료
try:
    subprocess.run(["pkill", "-f", "streamlit"], check=False)
    time.sleep(2)
except:
    pass

# 백그라운드에서 Streamlit 시작
streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "lab_helpers/lab_05/streamlit_app.py",
        "--server.port",
        "8501",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd=os.getcwd(),
)

time.sleep(5)

# 접속 가능한 URL 가져오기
streamlit_url = get_streamlit_url()

print("✅ Streamlit app launched!")
print("\n" + "=" * 80)
print(f"🌐 Access the chat interface at:\n{streamlit_url}")
print("=" * 80)
print("\n💡 Tips:")
print("   • The app uses the same gateway and authentication as the notebook")
print("   • Try queries like: 'What issues do you see in the CRM application?'")
print("   • Click 'Clear Chat History' in sidebar to reset conversation")
print("\n⚠️  To stop the app, run the cleanup cell below")

### 9.4 아키텍처: Streamlit 통합

```
┌─────────────────────────────────────────────────────────────────┐
│                    USER (Web Browser)                           │
│                    http://localhost:8501                        │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             │ HTTP
                             ↓
┌─────────────────────────────────────────────────────────────────┐
│                    STREAMLIT APP                                │
│  • Chat interface with streaming                                │
│  • Loads gateway_config.json                                    │
│  • Authenticates with Cognito                                   │
│  • Creates MCP client with JWT                                  │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             │ MCP + JWT
                             ↓
┌─────────────────────────────────────────────────────────────────┐
│                    CENTRAL GATEWAY                              │
│  • Validates JWT via interceptor                                │
│  • Routes to appropriate agent                                  │
└────────────────────────────┬────────────────────────────────────┘
                             │
                    ┌────────┴────────┬────────────────┐
                    ↓                 ↓                ↓
              Diagnostics       Remediation      Prevention
              (Lambda)          (Runtime)        (Runtime)
```

**주요 기능:**

- **실시간 스트리밍**: Agent 응답이 생성되는 즉시 확인합니다.
- **채팅 기록**: 대화 컨텍스트를 유지합니다.
- **도구 표시**: 호출 중인 도구를 사이드바에 표시합니다.
- **오류 처리**: 재시도 제안과 함께 이해하기 쉬운 오류 메시지를 제공합니다.
- **OAuth 통합**: Cognito를 통해 토큰을 자동으로 관리합니다.

### 9.5 Streamlit 앱 중지

In [ ]:
# Streamlit 프로세스 중지
try:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=5)
    print("✅ Streamlit app stopped")
except:
    subprocess.run(["pkill", "-f", "streamlit"], check=False)
    print("✅ Streamlit processes killed")

In [ ]:
# 포트 80과 8080에서 URL 시도
print(f"Click here to access the CRM App UI: '{get_app_url()}'")

## 10. 리소스 정리

In [ ]:
# Lab 05a 리소스만 정리
print("🗑️  Cleaning up...")

# ID에 'aiml301-central-gateway'가 포함된 Gateway 찾기
central_gateway_id = None
try:
    gateways = agentcore.list_gateways().get("items", [])
    for gateway in gateways:
        if "aiml301-central-gateway" in gateway.get("gatewayId", ""):
            central_gateway_id = gateway["gatewayId"]
            print(f"Found central gateway: {central_gateway_id}")
            break
except Exception as e:
    print(f"Error finding gateway: {e}")

if central_gateway_id:
    # 모든 Target을 먼저 삭제
    try:
        targets = agentcore.list_gateway_targets(gatewayIdentifier=central_gateway_id).get("items", [])
        print(f"Found {len(targets)} targets to delete")
        for target in targets:
            target_id = target["targetId"]
            print(f"Deleting target: {target_id}")
            agentcore.delete_gateway_target(gatewayIdentifier=central_gateway_id, targetId=target_id)
            time.sleep(2)
        print("✅ All targets deleted")
    except Exception as e:
        print(f"Error deleting targets: {e}")

    # Target이 완전히 삭제될 때까지 대기
    time.sleep(5)

    # Gateway 삭제
    try:
        print(f"Deleting gateway: {central_gateway_id}")
        response = agentcore.delete_gateway(gatewayIdentifier=central_gateway_id)
        print(f"Gateway delete response: {response}")
        time.sleep(3)
        print("✅ Gateway deleted")
    except Exception as e:
        print(f"❌ Error deleting gateway: {e}")
        import traceback

        traceback.print_exc()
else:
    print("⚠️  No gateway found with 'aiml301-central-gateway' in ID")

print("✅ Cleanup complete")